# Course notebook — Evaluation, Safety, and Governance

Build evidence that AI systems meet task, safety, security, compliance, and business requirements.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · The pilot that passed on average

Simone, a support operations lead, is asked to sign off an assistant that scores 96% on a broad test. Its mistakes concentrate on requests that change a customer’s eligibility. The launch sponsor sees a headline number. Simone sees a high-consequence decision being delegated without enough evidence.

The workplace scenario is fictional.


## Requirements and judgment


### Lecture 01 · Evaluation as Requirements

Turn desired behavior into tasks, cases, metrics, rubrics, slices, thresholds, and explicit failure tolerances.

#### A critical failure needs its own criterion

Routine requests dominate the report. A wrong eligibility answer can commit money while a wrong greeting is easy to repair.

Define severe failures and release gates before the overall average is calculated. Require adequate examples and clear review ownership for the high-consequence slice. A strong routine score alone cannot establish that slice is safe.

#### Read the mechanism

**task definitions**

Task definitions specify input, expected output, constraints, and graders for eval cases. Vague tasks produce noisy, incomparable metrics.

'Summarize ticket' becomes 'Extract product, issue, sentiment JSON matching schema X'.

**gold datasets**

Gold datasets hold authoritative labels or reference outputs for evaluation. They require versioning, access control, and refresh cadence.

200 lawyer-reviewed contract clauses with gold entity spans versioned quarterly.

**rubrics**

Rubrics score qualitative outputs against anchored criteria with examples at each level. They enable consistent human and LLM judging.

Support reply rubric scores correctness, completeness, tone, citations on 1–4 scale.

**slices**

Slices are subpopulations—language, tenant, risk tier—where aggregate metrics may hide failure.

95% overall accuracy can mask 60% on enterprise accounts.

**thresholds**

Thresholds are minimum acceptable metric values for release or routing decisions. They encode risk appetite numerically.

Faithfulness ≥ 0.92 and P0 safety 100% required for production promotion.

#### Worked example

**Situation:** A high-impact assistant may pass average quality while failing a safety-critical user slice—executive dashboards look green.

**Baseline:** Ten happy-path demo prompts as "eval."

**Application:** Derive 30 cases from real workflow risks (access grants, PII, abstention), assign rubrics, slices, pass thresholds, explicit tolerances for critical failures (zero tolerance on privilege escalation).

**Test cases:** (1) Normal: FAQ with citation. (2) Boundary: partial policy coverage. (3) Adversarial: combined injection plus privilege request.

**Measurement:** Pass rate overall and on critical slice, failure taxonomy, threshold gate decision.

**Design question:** Which case would you promote to blocking gate despite small sample size?

**Predict before running.** Evaluation is executable requirements for uncertain behavior. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 10.1: Evaluation as Requirements"""

CHAPTER = "10.1"
print("chapter hook:", CHAPTER)
cases = [
    {"id": 1, "input": "reset password", "must": "link to policy"},
    {"id": 2, "input": "delete tenant", "must": "require approval"},
]
for case in cases:
    print(case["id"], case["must"])
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Write a 30-case evaluation set from real workflow risks.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why is evaluation executable requirements for uncertain behavior?
2. How do slices differ from aggregate pass rates?
3. What eval baseline uses demo prompts only?

<details><summary>Check your reasoning</summary>

Q1: Tests encode must-hold behaviors with measurable pass/fail. Q2: Slices isolate populations where harm concentrates. Q3: Handful of cherry-picked successes.

</details>


### Lecture 02 · Metrics and Human Judgment

Combine exact metrics, semantic similarity, pairwise comparison, human review, LLM judges, calibration, and uncertainty.

#### Two reviewers read the same answer differently

One reviewer grades the policy conclusion, another grades clarity. Their mean scores agree and their individual ratings do not.

Write an observable rubric for each dimension and ask both reviewers to rate a small shared sample. Inspect disagreements and label cases independently before averaging.

#### Read the mechanism

**deterministic metrics**

Deterministic metrics—exact match, F1 on spans, JSON validity—give reproducible scores without sampling variance.

Schema validation pass rate is deterministic; helpfulness often is not.

**human evaluation**

Human evaluation labels outputs quality when automation cannot capture nuance or safety. Design for rater training, agreement, and throughput.

Lawyers label contract summaries for legal accuracy on 50 cases monthly.

**LLM judges**

LLM judges automate scoring using rubrics but must be calibrated against humans to avoid systematic bias.

GPT-4 judge scores faithfulness correlated 0.85 with human labels after calibration.

**confidence intervals**

Confidence intervals quantify uncertainty in metric estimates from finite eval sets. Comparing models requires overlapping intervals or formal tests.

Model A at 82% ± 3% versus Model B at 85% ± 4% may not be significantly different.

**inter-rater agreement**

Inter-rater agreement measures how consistently multiple human graders apply rubrics—Cohen's kappa, Krippendorff's alpha.

Low agreement on tone dimension means rubric needs refinement before scaling labeling.

#### Worked example

**Situation:** Automated metrics disagree with compliance reviewers on whether answers are "good enough."

**Baseline:** BLEU score on reference answers—misaligned with policy fidelity.

**Application:** Calibrate LLM judge against two human reviewers on 50 cases, measure inter-rater agreement, use pairwise comparisons for tie-breaks, report confidence intervals on pass rates.

**Test cases:** (1) Normal: all raters agree pass. (2) Boundary: judge-human disagreement. (3) Adversarial: judge favors fluent hallucination.

**Measurement:** Cohen's kappa judge-human, calibration drift over time, cost per human review hour.

**Design question:** When must human review override an automated judge pass?

**Predict before running.** Every metric encodes a theory of quality; validate that theory against real decisions. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 10.2: Metrics and Human Judgment"""

CHAPTER = "10.2"
print("chapter hook:", CHAPTER)
human = [1, 0, 1, 1]
judge = [1, 1, 1, 0]
agree = sum(h == j for h, j in zip(human, judge)) / len(human)
print({"agreement": round(agree, 2)})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Calibrate an automated judge against two human reviewers.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why does every metric encode a theory of quality?
2. How do confidence intervals change release decisions?
3. What metric baseline uses exact match on paraphrases?

<details><summary>Check your reasoning</summary>

Q1: BLEU rewards n-grams not fidelity or safety. Q2: Wide CI on small slice means pass rate uncertain—gate may fail. Q3: Exact string match on reference answers.

</details>


**Section checkpoint.** Choose an error whose cost justifies a separate release criterion and define who labels it.


## Stage evaluation and security


### Lecture 03 · Evaluation by System Stage

Evaluate ingestion, retrieval, generation, tools, agents, UX, latency, cost, and business outcomes separately and together.

#### Locate the failure before editing the prompt

An eligibility exception is correctly fetched but discarded during context assembly. The answer is then wrong despite a successful retrieval hit.

Measure retrieval, assembly, and final output separately. A trace that names the selected evidence can identify the broken stage. Freeze the failing example so a later prompt change cannot conceal the regression.

#### Read the mechanism

**component evals**

Component evals test retrieval, generation, tools, and UX stages independently before end-to-end runs. They localize failures.

Retrieval recall@10 evaluated separately from answer faithfulness on same queries.

**retrieval metrics**

Retrieval metrics—recall@k, MRR, nDCG—measure candidate set quality before generation sees it.

High recall@20 with poor faithfulness suggests generation issue, not retrieval.

**faithfulness**

Faithfulness checks that generated statements are entailed by retrieved evidence, not hallucinated additions. It is separate from fluency or user satisfaction.

Correct tone but wrong deductible amount is unfaithful despite readable prose.

**tool success**

Tool success rate tracks correct schema, auth, execution, and useful results from tool calls. It isolates integration failures from model reasoning.

60% tool success with high answer quality still blocks reliable agents.

**end-to-end evals**

End-to-end evals measure full pipeline outcomes on realistic inputs including latency and cost.

User question to cited answer passes only if retrieval, generation, and citation all succeed.

#### Worked example

**Situation:** RAG assistant fails in production; team argues whether retrieval, generation, or tools caused it.

**Baseline:** End-to-end thumbs-up/down only.

**Application:** Build failure attribution matrix: ingestion, retrieval recall, rerank, generation faithfulness, tool success, UX; run component evals with frozen downstream gold inputs.

**Test cases:** (1) Normal: retrieval fails, generation good. (2) Boundary: all components pass component tests but E2E fails interaction. (3) Adversarial: metric gaming by overfitting reranker to eval queries.

**Measurement:** Component pass rates, attributed failure percentage, fix validation on targeted slice.

**Design question:** Which component eval would you run first given wrong citations but right topic?

**Predict before running.** Stage-specific evaluation makes failures diagnosable and improvements attributable. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 10.3: Evaluation by System Stage"""

CHAPTER = "10.3"
print("chapter hook:", CHAPTER)
matrix = {"retrieval": 0.6, "rerank": 0.8, "generation": 0.9}
symptom = "wrong doc cited"
if symptom == "wrong doc cited":
    first = min(matrix, key=matrix.get)
print("investigate first:", first)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Build a failure attribution matrix for a RAG system.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why evaluate stages separately and together?
2. How does failure attribution guide fixes?
3. What baseline evaluates end-to-end only?

<details><summary>Check your reasoning</summary>

Q1: Isolates fixable boundaries vs interaction bugs. Q2: Low retrieval recall → fix index before tuning prompts. Q3: Single user satisfaction score.

</details>


### Lecture 04 · Security of AI Systems

Threat-model prompt injection, data exfiltration, tool abuse, identity confusion, insecure output handling, supply chain, and denial of service.

#### Treat the action boundary as part of the test

A support transcript contains untrusted text and the assistant can also retrieve restricted information. The workflow owner wants useful summaries without giving transcript text authority.

Keep read permissions and action approvals independent of generated content. A test should prove that the resulting action stays within the caller’s rights and that policy-relevant transcript text still reaches the summary.

#### Read the mechanism

**prompt injection**

Prompt injection embeds hostile instructions in untrusted content that models may follow instead of trusted policy.

A retrieved page saying 'ignore previous instructions' can redirect a summarizer to exfiltrate secrets.

**data exfiltration**

Data exfiltration via AI occurs when prompts or tools leak secrets, PII, or restricted docs to unauthorized parties.

Injection tricking model to dump system prompt or customer list into chat.

**tool abuse**

Tool abuse exploits excessive permissions—delete, send email, SQL write—through manipulated agent behavior.

Agent tricked into mass email via send_campaign tool with broad scope.

**sandboxing**

Sandboxing isolates code execution, browsing, or file access in restricted environments with network and filesystem limits.

Python tool runs in container without egress except allowlisted APIs.

**threat modeling**

Threat modeling systematically identifies assets, adversaries, and attack paths for AI systems—STRIDE, attack trees adapted for LLM risks.

Diagram data flow from user → retrieval → model → tools noting untrusted inputs.

#### Worked example

**Situation:** Red team attempts data exfiltration and tool abuse on the production assistant.

**Baseline:** Assume model safety training is sufficient.

**Application:** Threat model prompt injection, exfiltration via encoded output, tool argument injection, insecure output handling; run red-team suite, document mitigations and residual risk.

**Test cases:** (1) Normal: benign policy question. (2) Boundary: encoded secrets request. (3) Adversarial: chained injection through retrieved doc plus tool call.

**Measurement:** Successful attack count, mean time to detect, mitigation coverage map.

**Design question:** Which threat requires sandboxing tools rather than prompt hardening alone?

**Predict before running.** Treat models and retrieved content as untrusted components inside ordinary security boundaries. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 10.4: Security of AI Systems"""

CHAPTER = "10.4"
print("chapter hook:", CHAPTER)
attacks = ["inject retrieved", "exfil via markdown", "tool arg injection"]
mitigations = {"inject retrieved": "data labeling", "exfil via markdown": "output filter", "tool arg injection": "schema + sandbox"}
for a in attacks:
    print(a, "->", mitigations[a])
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Red-team a tool-enabled assistant and document mitigations.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why treat models and retrieved content as untrusted?
2. How does tool abuse differ from prompt injection?
3. What security baseline trusts the model?

<details><summary>Check your reasoning</summary>

Q1: Attackers influence inputs; boundaries must enforce policy. Q2: Injection manipulates intent; tool abuse executes effects. Q3: No red team, no output filtering.

</details>


**Section checkpoint.** Write separate checks for retrieving evidence and for using it correctly in the final response.


## Risk and governance


### Lecture 05 · Responsible AI and Risk

Assess bias, privacy, transparency, human impact, misuse, accessibility, high-impact decisions, and safe failure.

#### Measure who has to fix the misses

Two customer groups see the same response rate, but one group must contact support more often to resolve wrong answers. The extra work does not appear in the model score.

Track errors and correction burden per slice. Include an escalation route, collect representative cases, and assign ownership for decisions that a person can appeal.

#### Read the mechanism

**fairness**

Fairness examines disparate performance or harm across demographic or regional groups. Legal and ethical requirements vary by jurisdiction and use case.

Loan model approval rate disparity across groups triggers review even if aggregate AUC is high.

**privacy**

Privacy limits collection, retention, and exposure of personal data in training, logs, and outputs. GDPR and similar laws define user rights.

Support logs must redact credit card numbers; retention capped at 90 days.

**transparency**

Transparency discloses when users interact with AI, what data is used, and system limitations. It supports informed consent and trust.

Chat banner states AI-generated; citations show source documents.

**human oversight**

Human oversight defines when and how people supervise agents—monitoring dashboards, escalation queues, kill switches. It scales only with clear triggers.

Escalate to human when confidence < 0.7 or spend > $1 on a single task.

**impact assessment**

Impact assessment evaluates consequences of deploying AI on people, rights, and society before high-risk launch.

Automated hiring tool requires assessment of bias, appeal process, and human override.

#### Worked example

**Situation:** Assistant used for performance review summaries; HR worries about bias and privacy.

**Baseline:** Ship feature with generic "be fair" prompt line.

**Application:** Conduct impact assessment: affected populations, data minimization, transparency, human oversight for consequential outputs, accessibility, misuse scenarios, monitoring plan.

**Test cases:** (1) Normal: voluntary feedback summary. (2) Boundary: manager-only sensitive note. (3) Adversarial: inferring protected attributes from writing style.

**Measurement:** Bias slice metrics, privacy incident count, oversight compliance rate.

**Design question:** Which use case moves this feature into human-in-the-loop mandatory review?

**Predict before running.** Responsible AI is a lifecycle of decisions and evidence, not a one-time checklist. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 10.5: Responsible AI and Risk"""

CHAPTER = "10.5"
print("chapter hook:", CHAPTER)
use_cases = [
    {"name": "grammar fix", "impact": "low"},
    {"name": "promotion recommendation", "impact": "high"},
]
for uc in use_cases:
    hitl = uc["impact"] == "high"
    print(uc["name"], "human_review:", hitl)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Write an impact assessment for a consequential use case.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why is responsible AI a lifecycle not a checklist?
2. How does transparency differ from marketing trust badges?
3. What RAI baseline is a one-time legal sign-off?

<details><summary>Check your reasoning</summary>

Q1: Risks evolve with data, users, and integrations. Q2: Transparency shows limits and data use; badges claim virtue without evidence. Q3: Checkbox at launch with no monitoring.

</details>


### Lecture 06 · Governance and Assurance

Define ownership, inventory, risk tiers, policies, approvals, audit evidence, incidents, exceptions, vendor review, and retirement.

#### What would the reviewer need six months later?

The launch meeting authorizes a limited trial. A new manager cannot find which version was approved or which failures would require a pause.

Store the version, evidence, decision owner, allowed scope, and rollback conditions together. This is the record people need to investigate an outcome after the original reviewers have moved on.

#### Read the mechanism

**AI inventory**

AI inventory catalogs models, datasets, prompts, and features with owners, risk tier, and dependencies. You cannot govern what you cannot find.

Registry lists prod chatbot v3, embedding model e5-v2, fine-tune data v1.4 with owners.

**risk tiers**

Risk tiers classify AI systems by potential harm—low, medium, high—driving eval depth, approval path, and monitoring.

Internal summarization is tier 1; automated credit decision is tier 3 with full gate package.

**model cards**

Model cards document intended use, training data, limitations, metrics, and ethical considerations for a model version.

Card states model not for legal advice; lists languages supported and known failure modes.

**audit evidence**

Audit evidence collects eval reports, approvals, change logs, and incident records demonstrating controlled AI delivery.

Release ticket links eval v47 pass, security review, and canary metrics.

**incident response**

Incident response defines detect, triage, mitigate, communicate, and postmortem for AI failures—hallucination harm, data leak, outage.

Kill switch disables feature flag within 5 minutes of P0 safety incident.

#### Worked example

**Situation:** Company scales to 40 AI features; audits ask who owns risk acceptance for the onboarding assistant.

**Baseline:** Each team self-certifies with inconsistent evidence.

**Application:** Create lightweight governance model: AI inventory, risk tiers, required artifacts by tier, approval paths, incident process, vendor review, retirement criteria.

**Test cases:** (1) Normal: low-risk internal summarizer. (2) Boundary: medium-risk customer-facing bot. (3) Adversarial: shadow IT model deployed without inventory entry.

**Measurement:** Inventory coverage %, audit finding count, mean approval cycle time.

**Design question:** What artifact distinguishes tier-2 from tier-1 without bureaucratic overload?

**Predict before running.** Governance should make safe delivery easier by clarifying authority, evidence, and escalation. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 10.6: Governance and Assurance"""

CHAPTER = "10.6"
print("chapter hook:", CHAPTER)
tiers = {1: ["model card"], 2: ["model card", "eval report", "rollback plan"]}
feature = "onboarding assistant"
tier = 2
print({"required": tiers[tier]})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Create a lightweight governance operating model for a mid-size company.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. How should governance make safe delivery easier?
2. What belongs in an AI inventory entry?
3. What governance baseline is ad hoc per project?

<details><summary>Check your reasoning</summary>

Q1: Clear tiers, templates, and escalation reduce guesswork. Q2: Owner, data, model, evals, risk tier, approvals. Q3: No inventory, no standard evidence.

</details>


**Section checkpoint.** State who may approve release and what evidence or failure blocks it.


## Course workshop

Compare aggregate accuracy with accuracy on a critical slice. Add passing general cases and observe why the critical gate should remain closed.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Calculate overall and slice metrics with risk-specific release gates."""
CASES = [
    ("general", True), ("general", True), ("general", False),
    ("high-risk", True), ("high-risk", True),
]


def accuracy(rows): return sum(ok for _, ok in rows) / len(rows)
overall = accuracy(CASES)
slices = {name: accuracy([row for row in CASES if row[0] == name]) for name in {name for name, _ in CASES}}
release = overall >= 0.75 and slices["high-risk"] == 1.0
print("overall=", round(overall, 3), "slices=", slices, "release=", release)
assert (0 if release else 1) == 0


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
expanded = CASES + [("high-risk", False)] + [("general", True)] * 100
critical = [row for row in expanded if row[0] == "high-risk"]
assert accuracy(expanded) > 0.95
assert accuracy(critical) < 1.0
print('Workshop checks passed')


## Course project

Create an evaluation and assurance package with datasets, rubrics, adversarial tests, release gates, and governance evidence.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
